# FLEO + YOLOv12 on FER2013 — final pipeline

Trains FLEO end-to-end, runs the ablation (Table 2), and makes the paper figures.

### Before Run All:
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. **+ Add Input** → search `fer2013` (msambare) → Add
4. **Run All** (or run the cells top to bottom).

## 1. Get the project + install dependencies

In [ ]:
import os
BRANCH = 'claude/sproject-fleo-contents-whiqzn'
os.chdir('/kaggle/working')
!rm -rf FLEO
!git clone --branch $BRANCH https://github.com/olfa-askri/fleo.git FLEO
os.chdir('/kaggle/working/FLEO')

# ultralytics is preinstalled on Kaggle; the code falls back to yolo11 if
# YOLOv12 configs are absent, so the slow '-U' upgrade is not needed.
!pip install scikit-learn pyyaml thop -q
import ultralytics; print('ultralytics', ultralytics.__version__)

## 2. Sanity checks (math + unit tests)

In [ ]:
!python reference/numpy_reference.py
!python tests/test_fleo.py

## 3. Locate the FER2013 dataset

In [ ]:
import os
EMO = {'angry','anger','disgust','fear','happy','neutral','sad','surprise'}
def _is_emo(p):
    try:
        subs = {d.lower() for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))}
    except OSError:
        return False
    return len(subs & EMO) >= 4

FER_ROOT = None
for root, dirs, _ in os.walk('/kaggle/input'):
    if os.path.isdir(os.path.join(root, 'train')) and _is_emo(os.path.join(root, 'train')):
        FER_ROOT = root; break
    if _is_emo(root):
        FER_ROOT = root; break
assert FER_ROOT, 'FER2013 not found - add it via "+ Add Input" (search fer2013).'
print('FER2013 root:', FER_ROOT)

## 4. Smoke test (2 epochs)
Quick check that everything runs before the long training.

In [ ]:
!python train_fer2013.py --data "$FER_ROOT" --backbone yolov12s --epochs 2 --device cuda

## 5. Full training 🚀
YOLO backbone + FLEO block (Gram-Schmidt + SE + binding) trained with the
confusion-aware fuzzy loss. Best checkpoint → `checkpoints/best_fleo.pt`.

In [ ]:
!python train_fer2013.py --data "$FER_ROOT" --backbone yolov12s \
    --epochs 60 --batch-size 64 --lr 3e-4 --img-size 128 --device cuda

## 6. Ablation study → Table 2 🔬
baseline → +SE → +FLEO(no binding) → +FLEO(full) → +fuzzy, mean ± std.
Start small (`--epochs 5 --seeds 0`) to gauge runtime, then scale up.

In [ ]:
!python run_ablation.py --data "$FER_ROOT" --backbone yolov12s \
    --epochs 40 --batch-size 64 --seeds 0 1 2 --out results/ablation.json

## 7. Figures for the paper 📊
Confusion matrix (anger/sad, fear/surprise cells) + t-SNE of the features.

In [ ]:
!python evaluate.py --data "$FER_ROOT" --backbone yolov12s --mode fleo_full \
    --ckpt checkpoints/best_fleo.pt --out results
from IPython.display import Image, display
for fig in ['results/confusion_matrix.png', 'results/tsne.png']:
    display(Image(filename=fig))

## 8. Save artifacts (download from the Output tab)

In [ ]:
import shutil, os
if os.path.isdir('results'):
    shutil.make_archive('/kaggle/working/fleo_results', 'zip', 'results')
if os.path.exists('checkpoints/best_fleo.pt'):
    shutil.copy('checkpoints/best_fleo.pt', '/kaggle/working/best_fleo.pt')
print('Done. See the Output tab for fleo_results.zip and best_fleo.pt')